In [1]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set plot style
plt.style.use('seaborn-v0_8')

ModuleNotFoundError: No module named 'pandas'

## Data Loading
We search for `roofline.json` files in the subdirectories and extract the relevant metrics.

In [ ]:
def load_ert_results(base_dir='.'):
    results = []
    for root, dirs, files in os.walk(base_dir):
        if 'roofline.json' in files:
            filepath = os.path.join(root, 'roofline.json')
            try:
                with open(filepath, 'r') as f:
                    data = json.load(f)
                
                # Extract System Name from directory structure
                # Assuming structure: ./SystemName/Run.XXX/roofline.json
                parts = os.path.normpath(root).split(os.sep)
                if len(parts) >= 2:
                    system_name = parts[-2] # Parent of Run.XXX
                else:
                    system_name = "Unknown"

                # Extract GFLOPs
                gflops_data = data['empirical']['gflops']['data']
                peak_gflops = gflops_data[0][1] if gflops_data else 0

                # Extract Bandwidths
                gbytes_data = data['empirical']['gbytes']['data']
                bandwidths = {item[0]: item[1] for item in gbytes_data}

                results.append({
                    'System': system_name,
                    'Peak GFLOPs': peak_gflops,
                    'L1 Bandwidth': bandwidths.get('L1', 0),
                    'L2 Bandwidth': bandwidths.get('L2', 0),
                    'L3 Bandwidth': bandwidths.get('L3', 0),
                    'DRAM Bandwidth': bandwidths.get('DRAM', 0),
                    'Path': filepath
                })
            except Exception as e:
                print(f"Error reading {filepath}: {e}")
    
    return pd.DataFrame(results)

df = load_ert_results()
display(df)

## Peak Performance Comparison
This plot compares the empirical peak floating-point performance (GFLOPs) across different systems.
Higher is better. This metric represents the maximum theoretical compute capability achieved by the benchmark.

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(df['System'], df['Peak GFLOPs'], color='skyblue')
plt.title('Empirical Peak FP64 Performance')
plt.ylabel('GFLOPs')
plt.xlabel('System')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}',
             ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Memory Bandwidth Comparison
This plot compares the bandwidth of different memory hierarchy levels (L1, L2, L3, DRAM).
It highlights the speed difference between cache levels and main memory.
Note: Some systems might not have all cache levels detected or reported.

In [ ]:
# Melt the DataFrame for grouped bar chart
bandwidth_cols = ['L1 Bandwidth', 'L2 Bandwidth', 'L3 Bandwidth', 'DRAM Bandwidth']
df_melted = df.melt(id_vars=['System'], value_vars=bandwidth_cols, var_name='Memory Level', value_name='Bandwidth (GB/s)')

plt.figure(figsize=(12, 7))
import seaborn as sns
sns.barplot(data=df_melted, x='System', y='Bandwidth (GB/s)', hue='Memory Level')
plt.title('Memory Hierarchy Bandwidth Comparison')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Roofline Model
The Roofline model visualizes the performance limits of a system.
- **Horizontal lines** represent the Peak GFLOPs (Compute Bound).
- **Diagonal lines** represent the Peak Bandwidths (Memory Bound).
The "Knee points" (Ridge points) indicate the Arithmetic Intensity required to reach peak performance.

In [ ]:
def plot_roofline(system_row, ax):
    system = system_row['System']
    peak_gflops = system_row['Peak GFLOPs']
    bandwidths = {
        'L1': system_row['L1 Bandwidth'],
        'L2': system_row['L2 Bandwidth'],
        'L3': system_row['L3 Bandwidth'],
        'DRAM': system_row['DRAM Bandwidth']
    }
    
    # X-axis: Arithmetic Intensity (FLOPs/Byte)
    # Create a range of AI values (log scale)
    ai = np.logspace(-2, 4, 100)
    
    # Plot Ceilings
    # Compute Bound
    ax.axhline(y=peak_gflops, color='black', linestyle='-', label=f'Peak GFLOPs ({peak_gflops:.1f})')
    
    # Memory Bounds
    colors = {'L1': 'red', 'L2': 'green', 'L3': 'blue', 'DRAM': 'orange'}
    for mem, bw in bandwidths.items():
        if bw > 0:
            # Performance = min(Peak GFLOPs, Bandwidth * AI)
            perf = np.minimum(peak_gflops, bw * ai)
            ax.plot(ai, perf, label=f'{mem} BW ({bw:.1f} GB/s)', color=colors.get(mem, 'gray'))
            
            # Calculate Ridge Point (Intersection)
            ridge_ai = peak_gflops / bw
            ax.plot(ridge_ai, peak_gflops, 'o', color=colors.get(mem, 'gray'))
            ax.text(ridge_ai, peak_gflops * 1.1, f'{ridge_ai:.2f}', fontsize=8, ha='center')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Arithmetic Intensity (FLOPs/Byte)')
    ax.set_ylabel('Performance (GFLOPs)')
    ax.set_title(f'Roofline Model: {system}')
    ax.legend()
    ax.grid(True, which="both", ls="-", alpha=0.2)

# Create subplots for each system
num_systems = len(df)
cols = 2
rows = (num_systems + 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 6 * rows))
axes = axes.flatten()

for i, (index, row) in enumerate(df.iterrows()):
    plot_roofline(row, axes[i])

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Machine Balance
Machine Balance is defined as Peak Flops / Peak Bandwidth.
It represents the number of floating-point operations the processor can perform for every byte of data moved from memory.
A higher machine balance means the system is more "compute-heavy" and requires algorithms with higher arithmetic intensity to utilize the compute units.

In [ ]:
# Calculate Machine Balance for DRAM
df['Machine Balance (DRAM)'] = df['Peak GFLOPs'] / df['DRAM Bandwidth']

plt.figure(figsize=(10, 6))
bars = plt.bar(df['System'], df['Machine Balance (DRAM)'], color='purple')
plt.title('Machine Balance (DRAM)')
plt.ylabel('FLOPs / Byte')
plt.xlabel('System')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}',
             ha='center', va='bottom')

plt.tight_layout()
plt.show()